# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema available at:
[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

The dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements such as GAD-7, PHQ-9, and PSQ scores.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata is accessed as a single object (do not subscript or iterate)
meta = dataset.metadata
print(f"Dataset name:\n  {meta.name}\n\nDataset description:\n  {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets with their @id and name
print("Available record sets (by '@id' and name):\n")
record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets:
    display_str = f"@id: {rs['@id']}, name: {rs.get('name', '[No name]')}"
    print(display_str)
    record_set_ids.append(rs['@id'])

if not record_set_ids:
    print("No record sets found in the Croissant metadata. Please check the schema for actual entries.")

### Explore Fields in a Record Set
Choose a record set `@id` from the list above. For demonstration, we'll take the first found record set.

In [ ]:
# For demonstration, pick the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    # List fields (columns) for this record set
    print(f"\nFields for record set '@id': {first_rs_id}:\n")
    fields = None
    for rs in record_sets:
        if rs['@id'] == first_rs_id:
            fields = rs.get('fields', [])
            break
    if fields:
        for f in fields:
            print(f"@id: {f['@id']}, name: {f.get('name', '[No name]')}, dataType: {f.get('dataType', '[unknown]')}")
    else:
        print("No fields found for this record set.")
else:
    print("No record sets to explore.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Always reference each record set by its `@id`.

In [ ]:
# Aggregate all record set @ids
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set '@id': {rs_id} ...")
    try:
        # Load records using @id
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Failed to load records for @id: {rs_id}. Error: {e}\n")

# For analysis, keep reference to the first dataframe
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"First record set '@id': {main_record_set_id}")
    df_main = dataframes[main_record_set_id]
    print(f"\nSample of records:")
    display(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA steps. We will:
- Pick a numeric field by its `@id` (e.g., a GAD-7, PHQ-9, or PSQ score, or e.g., 'cr:phq_9_score')
- Filter records where the score exceeds a threshold (e.g., 10)
- Normalize the field
- Optionally group by another field (e.g., demographic info like 'cr:age' or 'cr:gender')

Note: Replace placeholders with the actual `@id` as found in the metadata overview cells above.

In [ ]:
# Example: Use 'cr:phq_9_score' and 'cr:gender' if present

# Replace with actual field @id from metadata if necessary
numeric_field = 'cr:phq_9_score'  # Example field '@id' for PHQ-9 score
group_field = 'cr:gender'         # Example @id for grouping

# Check that the field exists
df = dataframes[main_record_set_id]

if numeric_field in df.columns:
    # Remove nulls and ensure numeric type
    df_numeric = df[df[numeric_field].notnull()].copy()
    df_numeric[numeric_field] = pd.to_numeric(df_numeric[numeric_field], errors='coerce')

    # Filter
    threshold = 10
    filtered_df = df_numeric[df_numeric[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold}:\n")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}':\n")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by group_field if present
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':\n")
        display(grouped_df)
else:
    print(f"Field '@id' {numeric_field} not found in columns. Available columns: {df.columns.tolist()}")

## 5. Visualization
Visualize the distribution of the chosen numeric field (e.g., PHQ-9 scores) and, if available, compare groups (e.g., by gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of PHQ-9 scores
if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title('Distribution of PHQ-9 Score')
    plt.xlabel('PHQ-9 Score')
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by gender if exists
    if group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field], showmeans=True)
        plt.title('PHQ-9 Scores by Gender')
        plt.xlabel('Gender')
        plt.ylabel('PHQ-9 Score')
        plt.show()
else:
    print(f"Field '@id' {numeric_field} not found in columns. Available columns: {df.columns.tolist()}")

## 6. Conclusion

In this notebook, we've used `mlcroissant` to:
- Programmatically explore the structure of the dataset via metadata and `@id` fields
- Load and preview records from each record set using their `@id`
- Perform EDA such as filtering and normalizing a mental health score
- Visualize data distributions and group differences

For further exploration, consider:
- Testing survey completeness and missingness
- Cross-relating demographic fields with mental health outcomes
- Exporting cleaned data subsets for downstream analysis

**Refer to all logical fields by their `@id` as represented in the Croissant schema for maximum reproducibility and interoperability.**